# PcamVLM — Lightweight ResNet-18 Training (CPU, No GPU)

**Goal:** Train a ResNet-18 on your **laptop CPU**. No GPU needed.
Trained model saved to `models/lightweight-pcam.pt` for Streamlit app.

**What you will do:**
1. Install lightweight deps
2. Train on synthetic H&E patches (2000 samples, ~30-60s)
3. Test predictions
4. Save model + upload to GitHub
5. Deploy on Streamlit Cloud


## 1. Install Dependencies
Only: torch, torchvision, streamlit. No transformers/bitsandbytes.

In [ ]:
from pathlib import Path
pwd = Path.cwd()
assert (pwd/"app.py").exists(), "Run from PcamVLM/PcamVLM/"
print("Correct:", pwd)

In [ ]:
!pip install -r requirements-deploy.txt -q
import torch
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), "(False=OK, using CPU)")

## 2. Load Project Modules

In [ ]:
import sys; sys.path.insert(0,".")
from src import core as C, data as D, evaluate as E, lightweight as L
from src.lightweight import train_lightweight, predict_lightweight_pcam, load_lightweight_model
print("All modules imported")

## 3. View Sample Synthetic Images
Purple nuclei on pink cytoplasm. Tumor = denser/darker.

In [ ]:
import matplotlib.pyplot as plt
rows = D.make_synthetic_pcam(6, seed=42); fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for i, ax in enumerate(axes.flat):
    r = rows[i]; label = C.PCAM_LABEL_NAMES[r["label"]]
    ax.imshow(r["image"]); ax.set_title(label.upper(), fontsize=14, fontweight="bold",
                 color="red" if r["label"]==1 else "green"); ax.axis("off")
plt.suptitle("Synthetic PCam: Normal vs Tumor", fontsize=13); plt.tight_layout(); plt.show()

## 4. Train ResNet-18 (CPU, ~30-60s)
Frozen backbone, trained classifier head. 2000 samples, 5 epochs.

In [ ]:
import time; t0 = time.time()
train_lightweight(task="pcam", train_size=2000, val_size=500, epochs=5, batch_size=32, seed=42)
print(f"Done in {time.time()-t0:.1f}s")

## 5. Verify Saved Model

In [ ]:
import json
pt = Path("models/lightweight-pcam.pt"); js = Path("models/lightweight-pcam.json")
print(f"Weights: {pt.name} ({pt.stat().st_size/1e6:.1f} MB)")
meta = json.loads(js.read_text())
for k,v in meta.items(): print(f"  {k}: {v}")
print(f"Loadable: {load_lightweight_model("pcam") is not None}")

## 6. Test Predictions

In [ ]:
model = load_lightweight_model("pcam")
test = D.make_synthetic_pcam(10, seed=99)
fig, axes = plt.subplots(2, 5, figsize=(15, 6)); correct = 0
for i, ax in enumerate(axes.flat):
    r = test[i]; out = predict_lightweight_pcam(model, r["image"])
    ok = out["pred"] == r["label"]; correct += int(ok)
    ax.imshow(r["image"])
    ax.set_title(f"GT:{C.PCAM_LABEL_NAMES[r["label"]]}
Pred:{out["pred_name"]} (P={out["yes_prob"]:.0%})",
                 fontsize=10, color="green" if ok else "red"); ax.axis("off")
plt.suptitle(f"{correct}/10 correct", fontsize=14); plt.tight_layout(); plt.show()

In [ ]:
ev = D.make_synthetic_pcam(200, seed=77); t, p = [], []
for r in ev:
    out = predict_lightweight_pcam(model, r["image"])
    t.append(r["label"]); p.append(out["pred"])
rpt = E.evaluate_binary(t, p)
print(f"Accuracy: {rpt.accuracy:.2%}")
print(f"Sensitivity: {rpt.per_class[1]["recall"]:.2%}")
print(f"F1 Score: {rpt.per_class[1]["f1"]:.2%}")

## 7. Upload to GitHub + Deploy
```bash
git add PcamVLM/
git commit -m "Add PcamVLM with lightweight model"
git push
```

**Streamlit Cloud:** New app → `PcamVLM/PcamVLM/app.py`

Files uploaded: app.py, src/lightweight.py, models/lightweight-pcam.pt, etc.